In [0]:

# CLEANING, CASTING, DEDUPLICATION & QUARANTINE
from pyspark.sql import functions as F
from pyspark.sql.window import Window
from pyspark.sql.types import *

# UNITY CATALOG CONFIGURATION

CATALOG = "retail_demo"
RAW_SCHEMA = "raw"
SILVER_SCHEMA = "silver"

# PROJECT PATH

BASE_PATH = (
    "/Volumes/retail_demo/raw/retail_files/"
    "retail_delta_project"
)

print(f"Catalog       : {CATALOG}")
print(f"Raw Schema    : {RAW_SCHEMA}")
print(f"Silver Schema : {SILVER_SCHEMA}")
print(f"Base Path     : {BASE_PATH}")

print("=" * 70)

Catalog       : retail_demo
Raw Schema    : raw
Silver Schema : silver
Base Path     : /Volumes/retail_demo/raw/retail_files/retail_delta_project


In [0]:
#  VERIFY BRONZE SOURCE TABLES

bronze_tables = [
    "bronze_customers_incremental",
    "bronze_products_incremental",
    "bronze_orders_incremental"
]
for table_name in bronze_tables:

    full_table_name = f"{CATALOG}.{RAW_SCHEMA}.{table_name}"

    try:
        count = spark.table(full_table_name).count()
        print(f" {full_table_name:<55} {count:,} rows")

    except Exception as e:
        print(f" {full_table_name:<55} NOT FOUND")

 retail_demo.raw.bronze_customers_incremental            630 rows
 retail_demo.raw.bronze_products_incremental             180 rows
 retail_demo.raw.bronze_orders_incremental               6,135 rows


In [0]:
#  LOAD BRONZE TABLES

bronze_customers = spark.table(
    f"{CATALOG}.{RAW_SCHEMA}.bronze_customers_incremental"
)
bronze_products = spark.table(
    f"{CATALOG}.{RAW_SCHEMA}.bronze_products_incremental"
)
bronze_orders = spark.table(
    f"{CATALOG}.{RAW_SCHEMA}.bronze_orders_incremental"
)
print("Bronze tables loaded successfully.")
print("Customers:", bronze_customers.count())
print("Products :", bronze_products.count())
print("Orders   :", bronze_orders.count())

Bronze tables loaded successfully.
Customers: 630
Products : 180
Orders   : 6135


In [0]:
# INSPECT BRONZE SCHEMAS
print("\n CUSTOMERS ")
bronze_customers.printSchema()

print("\n PRODUCTS")
bronze_products.printSchema()

print("\n ORDERS ")
bronze_orders.printSchema()


 CUSTOMERS 
root
 |-- customer_id: string (nullable = true)
 |-- customer_name: string (nullable = true)
 |-- city: string (nullable = true)
 |-- segment: string (nullable = true)
 |-- gender: string (nullable = true)
 |-- signup_date: string (nullable = true)
 |-- status: string (nullable = true)
 |-- effective_date: string (nullable = true)
 |-- operation: string (nullable = true)
 |-- order_id: string (nullable = true)
 |-- order_ts: string (nullable = true)
 |-- product_id: string (nullable = true)
 |-- store_id: string (nullable = true)
 |-- quantity: string (nullable = true)
 |-- unit_price: string (nullable = true)
 |-- discount_pct: string (nullable = true)
 |-- gross_amount: string (nullable = true)
 |-- payment_method: string (nullable = true)
 |-- order_status: string (nullable = true)
 |-- ingest_date: string (nullable = true)
 |-- product_name: string (nullable = true)
 |-- category: string (nullable = true)
 |-- brand: string (nullable = true)
 |-- created_date: string (

In [0]:
# CLEAN CUSTOMERS

from pyspark.sql import functions as F
silver_customers = (
    bronze_customers
    .select(
        "customer_id",
        "customer_name",
        "city",
        "segment",
        "gender",
        "signup_date",
        "status",
        "effective_date",
        "operation",
        "ingest_ts",
        "load_type",
        "source_file"
    )
    # Clean string columns
    .withColumn("customer_id", F.trim(F.col("customer_id")))
    .withColumn("customer_name", F.trim(F.col("customer_name")))
    .withColumn("city", F.trim(F.col("city")))
    .withColumn("segment", F.trim(F.col("segment")))
    .withColumn("gender", F.trim(F.col("gender")))
    .withColumn("status", F.trim(F.col("status")))
    .withColumn("operation", F.trim(F.col("operation")))
    .withColumn(
        "signup_date",
        F.coalesce(
            F.expr("try_to_date(trim(signup_date), 'yyyy-MM-dd')"),
            F.expr("try_to_date(trim(signup_date), 'dd-MM-yyyy')")
        )
    )
    .withColumn(
        "effective_date",
        F.coalesce(
            F.expr("try_to_date(trim(effective_date), 'yyyy-MM-dd')"),
            F.expr("try_to_date(trim(effective_date), 'dd-MM-yyyy')")
        )
    )

    # One latest/unique record per customer_id
    .dropDuplicates(["customer_id"])
)

print("Customers Silver cleaning completed.")
print("Rows:", silver_customers.count())

silver_customers.printSchema()

Customers Silver cleaning completed.
Rows: 582
root
 |-- customer_id: string (nullable = true)
 |-- customer_name: string (nullable = true)
 |-- city: string (nullable = true)
 |-- segment: string (nullable = true)
 |-- gender: string (nullable = true)
 |-- signup_date: date (nullable = true)
 |-- status: string (nullable = true)
 |-- effective_date: date (nullable = true)
 |-- operation: string (nullable = true)
 |-- ingest_ts: timestamp (nullable = true)
 |-- load_type: string (nullable = true)
 |-- source_file: string (nullable = true)



In [0]:
# VALIDATE SILVER CUSTOMER
total_rows = silver_customers.count()
distinct_ids = (
    silver_customers
    .select("customer_id")
    .distinct()
    .count()
)
null_customer_ids = (
    silver_customers
    .filter(
        F.col("customer_id").isNull() |
        (F.trim(F.col("customer_id")) == "")
    )
    .count()
)
duplicate_customer_ids = (
    silver_customers
    .groupBy("customer_id")
    .count()
    .filter(F.col("count") > 1)
    .count()
)
null_signup_dates = (
    silver_customers
    .filter(F.col("signup_date").isNull())
    .count()
)
null_effective_dates = (
    silver_customers
    .filter(F.col("effective_date").isNull())
    .count()
)
print("Total rows:", total_rows)
print("Distinct customer IDs:", distinct_ids)
print("Null/empty customer IDs:", null_customer_ids)
print("Duplicate customer IDs:", duplicate_customer_ids)
print("NULL signup dates:", null_signup_dates)
print("NULL effective dates:", null_effective_dates)
display(silver_customers.limit(10))

Total rows: 582
Distinct customer IDs: 582
Null/empty customer IDs: 0
Duplicate customer IDs: 0
NULL signup dates: 4
NULL effective dates: 1


customer_id,customer_name,city,segment,gender,signup_date,status,effective_date,operation,ingest_ts,load_type,source_file
C00606,Customer_606,Kolkata,Silver,Other,2025-10-09,active,null,UPDATE,2026-08-11T18:10:49.962Z,incremental,/Volumes/retail_demo/raw/retail_files/retail_delta_project/datasets/incremental/day_2026-04-25/customers_cdc_2026-04-25.csv
C00527,Customer_527,Bengaluru,Regular,M,2024-04-10,inactive,2026-04-25,UPDATE,2026-08-11T18:10:49.962Z,incremental,/Volumes/retail_demo/raw/retail_files/retail_delta_project/datasets/incremental/day_2026-04-25/customers_cdc_2026-04-25.csv
C00660,Customer_660,Jaipur,Regular,F,2024-10-16,active,2026-04-25,UPDATE,2026-08-11T18:10:49.962Z,incremental,/Volumes/retail_demo/raw/retail_files/retail_delta_project/datasets/incremental/day_2026-04-25/customers_cdc_2026-04-25.csv
C01018,Customer_1018,Jaipur,Platinum,F,2025-04-06,active,2026-04-25,UPDATE,2026-08-11T18:10:49.962Z,incremental,/Volumes/retail_demo/raw/retail_files/retail_delta_project/datasets/incremental/day_2026-04-25/customers_cdc_2026-04-25.csv
C00781,Customer_781,Pune,Regular,F,2025-12-01,active,2026-04-25,UPDATE,2026-08-11T18:10:49.962Z,incremental,/Volumes/retail_demo/raw/retail_files/retail_delta_project/datasets/incremental/day_2026-04-25/customers_cdc_2026-04-25.csv
C02068,Customer_2068,Delhi,Silver,M,2024-10-14,active,2026-04-25,UPDATE,2026-08-11T18:10:49.962Z,incremental,/Volumes/retail_demo/raw/retail_files/retail_delta_project/datasets/incremental/day_2026-04-25/customers_cdc_2026-04-25.csv
C02152,Customer_2152,Hyderabad,Gold,M,2024-05-30,active,2026-04-25,UPDATE,2026-08-11T18:10:49.962Z,incremental,/Volumes/retail_demo/raw/retail_files/retail_delta_project/datasets/incremental/day_2026-04-25/customers_cdc_2026-04-25.csv
C01200,Customer_1200,Delhi,Gold,F,2024-03-30,active,2026-04-25,UPDATE,2026-08-11T18:10:49.962Z,incremental,/Volumes/retail_demo/raw/retail_files/retail_delta_project/datasets/incremental/day_2026-04-25/customers_cdc_2026-04-25.csv
C00959,Customer_959,Ahmedabad,Platinum,F,2025-09-02,active,2026-04-25,UPDATE,2026-08-11T18:10:49.962Z,incremental,/Volumes/retail_demo/raw/retail_files/retail_delta_project/datasets/incremental/day_2026-04-25/customers_cdc_2026-04-25.csv
C00439,Customer_439,Delhi,Platinum,Other,2025-08-10,active,2026-04-25,UPDATE,2026-08-11T18:10:49.962Z,incremental,/Volumes/retail_demo/raw/retail_files/retail_delta_project/datasets/incremental/day_2026-04-25/customers_cdc_2026-04-25.csv


In [0]:
#  CUSTOMERS QUARANTINE

customer_quarantine = (
    bronze_customers
    .select(
        "customer_id",
        "customer_name",
        "city",
        "segment",
        "gender",
        "signup_date",
        "status",
        "effective_date",
        "operation",
        "ingest_ts",
        "load_type",
        "source_file"
    )
    .withColumn("customer_id", F.trim(F.col("customer_id")))
    .withColumn("signup_date_raw", F.col("signup_date"))
    .withColumn("effective_date_raw", F.col("effective_date"))
    .withColumn(
        "signup_date_parsed",
        F.coalesce(
            F.expr("try_to_date(trim(signup_date), 'yyyy-MM-dd')"),
            F.expr("try_to_date(trim(signup_date), 'dd-MM-yyyy')")
        )
    )
    .withColumn(
        "effective_date_parsed",
        F.coalesce(
            F.expr("try_to_date(trim(effective_date), 'yyyy-MM-dd')"),
            F.expr("try_to_date(trim(effective_date), 'dd-MM-yyyy')")
        )
    )
    .filter(
        F.col("customer_id").isNull()
        | (F.trim(F.col("customer_id")) == "")
        | F.col("signup_date_parsed").isNull()
        | F.col("effective_date_parsed").isNull()
    )
)
print("Customer quarantine records:", customer_quarantine.count())
display(customer_quarantine)

Customer quarantine records: 5


customer_id,customer_name,city,segment,gender,signup_date,status,effective_date,operation,ingest_ts,load_type,source_file,signup_date_raw,effective_date_raw,signup_date_parsed,effective_date_parsed
C00606,Customer_606,Kolkata,Silver,Other,2025-10-09,active,not_a_date,UPDATE,2026-08-11T18:10:49.962Z,incremental,/Volumes/retail_demo/raw/retail_files/retail_delta_project/datasets/incremental/day_2026-04-25/customers_cdc_2026-04-25.csv,2025-10-09,not_a_date,2025-10-09,null
C00423,Customer_423,Jaipur,Gold,Other,31-31-2025,inactive,2026-04-25,UPDATE,2026-08-11T18:10:49.962Z,incremental,/Volumes/retail_demo/raw/retail_files/retail_delta_project/datasets/incremental/day_2026-04-25/customers_cdc_2026-04-25.csv,31-31-2025,2026-04-25,null,2026-04-25
C01422,Customer_1422,Chennai,Silver,M,31-31-2025,active,2026-04-25,UPDATE,2026-08-11T18:10:49.962Z,incremental,/Volumes/retail_demo/raw/retail_files/retail_delta_project/datasets/incremental/day_2026-04-25/customers_cdc_2026-04-25.csv,31-31-2025,2026-04-25,null,2026-04-25
C00066,Customer_66,Bengaluru,Regular,Other,31-31-2025,active,2026-04-24,UPDATE,2026-08-11T18:10:49.962Z,incremental,/Volumes/retail_demo/raw/retail_files/retail_delta_project/datasets/incremental/day_2026-04-24/customers_cdc_2026-04-24.csv,31-31-2025,2026-04-24,null,2026-04-24
C00710,Customer_710,Kolkata,Silver,M,31-31-2025,active,2026-04-26,UPDATE,2026-08-11T18:10:49.962Z,incremental,/Volumes/retail_demo/raw/retail_files/retail_delta_project/datasets/incremental/day_2026-04-26/customers_cdc_2026-04-26.csv,31-31-2025,2026-04-26,null,2026-04-26


In [0]:
# WRITE CLEAN CUSTOMERS TO SILVER

silver_customers_final = (
    silver_customers
    .filter(
        F.col("customer_id").isNotNull()
        & F.col("signup_date").isNotNull()
        & F.col("effective_date").isNotNull()
    )
)
(
    silver_customers_final
    .write
    .format("delta")
    .mode("overwrite")
    .option("overwriteSchema", "true")
    .saveAsTable(
        f"{CATALOG}.{SILVER_SCHEMA}.silver1_customers_clean"
    )
)
print("Customer Silver table written successfully.")
display(
    spark.sql(f"""
        SELECT COUNT(*) AS row_count
        FROM {CATALOG}.{SILVER_SCHEMA}.silver1_customers_clean
    """)
)

Customer Silver table written successfully.


row_count
577


In [0]:
#CLEAN PRODUCTS

silver_products = (
    bronze_products
    .select(
        "product_id",
        "product_name",
        "category",
        "brand",
        "unit_price",
        "status",
        "created_date",
        "effective_date",
        "operation",
        "ingest_ts",
        "load_type",
        "source_file"
    )
    .withColumn("product_id", F.trim(F.col("product_id")))
    .withColumn("product_name", F.trim(F.col("product_name")))
    .withColumn("category", F.trim(F.col("category")))
    .withColumn("brand", F.trim(F.col("brand")))
    .withColumn("status", F.trim(F.col("status")))
    .withColumn("operation", F.trim(F.col("operation")))
    .withColumn(
        "unit_price_clean",
        F.regexp_extract(
            F.col("unit_price"),
            r"[-+]?\d*\.?\d+",
            0
        )
    )
    .withColumn(
        "unit_price",
        F.expr("try_cast(unit_price_clean AS DECIMAL(18,2))")
    )
    .drop("unit_price_clean")
    .withColumn(
        "created_date",
        F.coalesce(
            F.expr("try_to_date(trim(created_date), 'yyyy-MM-dd')"),
            F.expr("try_to_date(trim(created_date), 'dd-MM-yyyy')")
        )
    )
    .withColumn(
        "effective_date",
        F.coalesce(
            F.expr("try_to_date(trim(effective_date), 'yyyy-MM-dd')"),
            F.expr("try_to_date(trim(effective_date), 'dd-MM-yyyy')")
        )
    )
    .withColumn(
        "category",
        F.when(
            F.col("category").isNull() |
            (F.trim(F.col("category")) == ""),
            F.lit("Unknown")
        ).otherwise(F.col("category"))
    )
    .withColumn(
        "brand",
        F.when(
            F.col("brand").isNull() |
            (F.trim(F.col("brand")) == ""),
            F.lit("Unknown")
        ).otherwise(F.col("brand"))
    )
    .withColumn(
        "status",
        F.when(
            F.col("status").isNull() |
            (F.trim(F.col("status")) == ""),
            F.lit("Unknown")
        ).otherwise(F.col("status"))
    )
    # One record per product
    .dropDuplicates(["product_id"])
)

print("Products Silver cleaning completed.")
print("Rows:", silver_products.count())

silver_products.printSchema()

Products Silver cleaning completed.
Rows: 166
root
 |-- product_id: string (nullable = true)
 |-- product_name: string (nullable = true)
 |-- category: string (nullable = true)
 |-- brand: string (nullable = true)
 |-- unit_price: decimal(18,2) (nullable = true)
 |-- status: string (nullable = true)
 |-- created_date: date (nullable = true)
 |-- effective_date: date (nullable = true)
 |-- operation: string (nullable = true)
 |-- ingest_ts: timestamp (nullable = true)
 |-- load_type: string (nullable = true)
 |-- source_file: string (nullable = true)



In [0]:

# VALIDATE SILVER PRODUCTS

total_rows = silver_products.count()
distinct_product_ids = (
    silver_products
    .select("product_id")
    .distinct()
    .count()
)
null_product_ids = (
    silver_products
    .filter(
        F.col("product_id").isNull() |
        (F.trim(F.col("product_id")) == "")
    )
    .count()
)
duplicate_product_ids = (
    silver_products
    .groupBy("product_id")
    .count()
    .filter(F.col("count") > 1)
    .count()
)
null_unit_prices = (
    silver_products
    .filter(F.col("unit_price").isNull())
    .count()
)
null_created_dates = (
    silver_products
    .filter(F.col("created_date").isNull())
    .count()
)
null_effective_dates = (
    silver_products
    .filter(F.col("effective_date").isNull())
    .count()
)
print("Total rows:", total_rows)
print("Distinct product IDs:", distinct_product_ids)
print("Null/empty product IDs:", null_product_ids)
print("Duplicate product IDs:", duplicate_product_ids)
print("NULL unit prices:", null_unit_prices)
print("NULL created dates:", null_created_dates)
print("NULL effective dates:", null_effective_dates)
display(silver_products.limit(10))

Total rows: 166
Distinct product IDs: 166
Null/empty product IDs: 0
Duplicate product IDs: 0
NULL unit prices: 0
NULL created dates: 0
NULL effective dates: 0


product_id,product_name,category,brand,unit_price,status,created_date,effective_date,operation,ingest_ts,load_type,source_file
P00173,Speaker 173,Electronics,BrandA,50560.38,discontinued,2025-09-11,2026-04-25,UPDATE,2026-08-11T17:55:36.528Z,incremental,/Volumes/retail_demo/raw/retail_files/retail_delta_project/datasets/incremental/day_2026-04-25/products_cdc_2026-04-25.csv
P00680,Perfume 680,Beauty,BrandC,40368.75,active,2024-12-24,2026-04-25,UPDATE,2026-08-11T17:55:36.528Z,incremental,/Volumes/retail_demo/raw/retail_files/retail_delta_project/datasets/incremental/day_2026-04-25/products_cdc_2026-04-25.csv
P00095,Smartphone 95,Electronics,BrandB,685.97,active,2023-10-15,2026-04-25,UPDATE,2026-08-11T17:55:36.528Z,incremental,/Volumes/retail_demo/raw/retail_files/retail_delta_project/datasets/incremental/day_2026-04-25/products_cdc_2026-04-25.csv
P00664,Cream 664,Beauty,BrandB,43522.35,discontinued,2023-03-22,2026-04-25,UPDATE,2026-08-11T17:55:36.528Z,incremental,/Volumes/retail_demo/raw/retail_files/retail_delta_project/datasets/incremental/day_2026-04-25/products_cdc_2026-04-25.csv
P00733,Jacket 733,Fashion,BrandA,26764.31,active,2023-07-31,2026-04-25,UPDATE,2026-08-11T17:55:36.528Z,incremental,/Volumes/retail_demo/raw/retail_files/retail_delta_project/datasets/incremental/day_2026-04-25/products_cdc_2026-04-25.csv
P00082,Jacket 82,Fashion,BrandA,52340.68,active,2025-01-26,2026-04-25,UPDATE,2026-08-11T17:55:36.528Z,incremental,/Volumes/retail_demo/raw/retail_files/retail_delta_project/datasets/incremental/day_2026-04-25/products_cdc_2026-04-25.csv
P00541,Laptop 541,Electronics,BrandA,77208.10,active,2025-12-22,2026-04-25,UPDATE,2026-08-11T17:55:36.528Z,incremental,/Volumes/retail_demo/raw/retail_files/retail_delta_project/datasets/incremental/day_2026-04-25/products_cdc_2026-04-25.csv
P00251,Jeans 251,Fashion,BrandA,88637.25,active,2025-11-28,2026-04-25,UPDATE,2026-08-11T17:55:36.528Z,incremental,/Volumes/retail_demo/raw/retail_files/retail_delta_project/datasets/incremental/day_2026-04-25/products_cdc_2026-04-25.csv
P00282,Chair 282,Home,BrandC,42367.76,active,2023-06-03,2026-04-25,UPDATE,2026-08-11T17:55:36.528Z,incremental,/Volumes/retail_demo/raw/retail_files/retail_delta_project/datasets/incremental/day_2026-04-25/products_cdc_2026-04-25.csv
P00142,Keyboard 142,Electronics,BrandC,16441.76,active,2025-06-06,2026-04-25,UPDATE,2026-08-11T17:55:36.528Z,incremental,/Volumes/retail_demo/raw/retail_files/retail_delta_project/datasets/incremental/day_2026-04-25/products_cdc_2026-04-25.csv


In [0]:
#  WRITE CLEAN PRODUCTS TO SILVER

(
    silver_products
    .write
    .format("delta")
    .mode("overwrite")
    .option("overwriteSchema", "true")
    .saveAsTable(
        f"{CATALOG}.{SILVER_SCHEMA}.silver1_products_clean"
    )
)
print("Product Silver table written successfully.")

display(
    spark.sql(f"""
        SELECT COUNT(*) AS row_count
        FROM {CATALOG}.{SILVER_SCHEMA}.silver1_products_clean
    """)
)

Product Silver table written successfully.


row_count
166


In [0]:
# INSPECT ORDERS DATA QUALITY

print("\nTotal Bronze Orders:", bronze_orders.count())

print(
    "Distinct order IDs:",
    bronze_orders.select("order_id").distinct().count()
)

print(
    "Null/empty order IDs:",
    bronze_orders.filter(
        F.col("order_id").isNull() |
        (F.trim(F.col("order_id")) == "")
    ).count()
)

print(
    "Null customer IDs:",
    bronze_orders.filter(
        F.col("customer_id").isNull() |
        (F.trim(F.col("customer_id")) == "")
    ).count()
)

print(
    "Null product IDs:",
    bronze_orders.filter(
        F.col("product_id").isNull() |
        (F.trim(F.col("product_id")) == "")
    ).count()
)

print(
    "Null store IDs:",
    bronze_orders.filter(
        F.col("store_id").isNull() |
        (F.trim(F.col("store_id")) == "")
    ).count()
)

print(
    "Null quantities:",
    bronze_orders.filter(
        F.col("quantity").isNull()
    ).count()
)

print(
    "Null unit prices:",
    bronze_orders.filter(
        F.col("unit_price").isNull() |
        (F.trim(F.col("unit_price")) == "")
    ).count()
)

print(
    "Null gross amounts:",
    bronze_orders.filter(
        F.col("gross_amount").isNull() |
        (F.trim(F.col("gross_amount")) == "")
    ).count()
)

print("\nSample raw orders:")
display(bronze_orders.limit(20))


Total Bronze Orders: 6135
Distinct order IDs: 6000
Null/empty order IDs: 0
Null customer IDs: 0
Null product IDs: 60
Null store IDs: 0
Null quantities: 0
Null unit prices: 0
Null gross amounts: 0

Sample raw orders:


order_id,order_ts,customer_id,product_id,store_id,quantity,unit_price,discount_pct,gross_amount,payment_method,order_status,ingest_date,coupon_code,_rescued_data,ingest_ts,load_type,source_file
OI3000001,2026-04-26T22:42:00.000Z,C01830,P00296,S056,5,64201.56,0.1,256806.24,CARD,returned,2026-04-26,null,null,2026-08-11T17:49:51.127Z,incremental,/Volumes/retail_demo/raw/retail_files/retail_delta_project/datasets/incremental/day_2026-04-26/orders_incremental_2026-04-26.csv
OI3000002,2026-04-26T04:25:00.000Z,C01705,P00334,S030,2,16032.79,0.0,25652.46,NETBANKING,cancelled,2026-04-26,null,null,2026-08-11T17:49:51.127Z,incremental,/Volumes/retail_demo/raw/retail_files/retail_delta_project/datasets/incremental/day_2026-04-26/orders_incremental_2026-04-26.csv
OI3000003,2026-04-23T00:00:00.000Z,C01378,P00014,S045,6,24907.63,0.1,141973.49,CARD,shipped,2026-04-26,NEW10,null,2026-08-11T17:49:51.127Z,incremental,/Volumes/retail_demo/raw/retail_files/retail_delta_project/datasets/incremental/day_2026-04-26/orders_incremental_2026-04-26.csv
OI3000004,2026-04-26T14:17:00.000Z,C02220,P00747,S030,2,61937.51,0.05,117681.27,CARD,shipped,2026-04-26,NEW10,null,2026-08-11T17:49:51.127Z,incremental,/Volumes/retail_demo/raw/retail_files/retail_delta_project/datasets/incremental/day_2026-04-26/orders_incremental_2026-04-26.csv
OI3000005,2026-04-26T14:51:00.000Z,C01276,P00513,S005,4,72010.17,0.05,273638.65,UPI,delivered,2026-04-26,null,null,2026-08-11T17:49:51.127Z,incremental,/Volumes/retail_demo/raw/retail_files/retail_delta_project/datasets/incremental/day_2026-04-26/orders_incremental_2026-04-26.csv
OI3000006,2026-04-26T21:59:00.000Z,C01186,P00747,S044,6,61937.51,0.2,297300.05,CARD,cancelled,2026-04-26,null,null,2026-08-11T17:49:51.127Z,incremental,/Volumes/retail_demo/raw/retail_files/retail_delta_project/datasets/incremental/day_2026-04-26/orders_incremental_2026-04-26.csv
OI3000007,2026-04-26T06:29:00.000Z,C02099,P00522,S026,4,7540.18,0.1,30160.72,NETBANKING,shipped,2026-04-26,null,null,2026-08-11T17:49:51.127Z,incremental,/Volumes/retail_demo/raw/retail_files/retail_delta_project/datasets/incremental/day_2026-04-26/orders_incremental_2026-04-26.csv
OI3000008,2026-04-26T00:01:00.000Z,C01410,P00432,S011,1,7269.25,0.2,5815.4,UPI,returned,2026-04-26,null,null,2026-08-11T17:49:51.127Z,incremental,/Volumes/retail_demo/raw/retail_files/retail_delta_project/datasets/incremental/day_2026-04-26/orders_incremental_2026-04-26.csv
OI3000009,2026-04-26T16:42:00.000Z,C00774,P00672,S032,2,48105.37,0.05,96210.74,CARD,delivered,2026-04-26,null,null,2026-08-11T17:49:51.127Z,incremental,/Volumes/retail_demo/raw/retail_files/retail_delta_project/datasets/incremental/day_2026-04-26/orders_incremental_2026-04-26.csv
OI3000010,2026-04-26T13:03:00.000Z,C02274,P00686,S006,1,81005.35,0.05,76955.08,CARD,cancelled,2026-04-26,null,null,2026-08-11T17:49:51.127Z,incremental,/Volumes/retail_demo/raw/retail_files/retail_delta_project/datasets/incremental/day_2026-04-26/orders_incremental_2026-04-26.csv


In [0]:
# CLEAN ORDERS


silver_orders = (
    bronze_orders
    .select(
        "order_id",
        "order_ts",
        "customer_id",
        "product_id",
        "store_id",
        "quantity",
        "unit_price",
        "discount_pct",
        "gross_amount",
        "payment_method",
        "order_status",
        "ingest_date",
        "coupon_code",
        "ingest_ts",
        "load_type",
        "source_file"
    )

    .withColumn("order_id", F.trim(F.col("order_id")))
    .withColumn("customer_id", F.trim(F.col("customer_id")))
    .withColumn("product_id", F.trim(F.col("product_id")))
    .withColumn("store_id", F.trim(F.col("store_id")))
    .withColumn(
        "payment_method",
        F.when(
            F.col("payment_method").isNull() |
            (F.trim(F.col("payment_method")) == ""),
            F.lit("Unknown")
        ).otherwise(F.trim(F.col("payment_method")))
    )
    .withColumn(
        "order_status",
        F.when(
            F.col("order_status").isNull() |
            (F.trim(F.col("order_status")) == ""),
            F.lit("Unknown")
        ).otherwise(F.trim(F.col("order_status")))
    )
    .withColumn(
        "unit_price_clean",
        F.regexp_extract(
            F.trim(F.col("unit_price")),
            r"[-+]?\d*\.?\d+",
            0
        )
    )
    .withColumn(
        "unit_price",
        F.expr(
            "try_cast(unit_price_clean AS DECIMAL(18,2))"
        )
    )
    .drop("unit_price_clean")

    .withColumn(
        "gross_amount_clean",
        F.regexp_extract(
            F.trim(F.col("gross_amount")),
            r"[-+]?\d*\.?\d+",
            0
        )
    )
    .withColumn(
        "gross_amount",
        F.expr(
            "try_cast(gross_amount_clean AS DECIMAL(18,2))"
        )
    )
    .drop("gross_amount_clean")
    .withColumn(
        "rn",
        F.row_number().over(
            Window
            .partitionBy("order_id")
            .orderBy(F.col("ingest_ts").desc())
        )
    )
    .filter(F.col("rn") == 1)
    .drop("rn")
)

print("Orders Silver cleaning completed.")
print("Rows after deduplication:", silver_orders.count())

silver_orders.printSchema()

Orders Silver cleaning completed.
Rows after deduplication: 6000
root
 |-- order_id: string (nullable = true)
 |-- order_ts: timestamp (nullable = true)
 |-- customer_id: string (nullable = true)
 |-- product_id: string (nullable = true)
 |-- store_id: string (nullable = true)
 |-- quantity: integer (nullable = true)
 |-- unit_price: decimal(18,2) (nullable = true)
 |-- discount_pct: double (nullable = true)
 |-- gross_amount: decimal(18,2) (nullable = true)
 |-- payment_method: string (nullable = true)
 |-- order_status: string (nullable = true)
 |-- ingest_date: date (nullable = true)
 |-- coupon_code: string (nullable = true)
 |-- ingest_ts: timestamp (nullable = true)
 |-- load_type: string (nullable = true)
 |-- source_file: string (nullable = true)



In [0]:
# ORDERS QUARANTINE

orders_quality_check = (
    silver_orders
    .withColumn(
        "order_id_clean",
        F.trim(F.col("order_id"))
    )
    .withColumn(
        "customer_id_clean",
        F.trim(F.col("customer_id"))
    )
    .withColumn(
        "product_id_clean",
        F.trim(F.col("product_id"))
    )
    .withColumn(
        "store_id_clean",
        F.trim(F.col("store_id"))
    )
)
order_quarantine = (
    orders_quality_check
    .filter(
        F.col("order_id_clean").isNull()
        | (F.col("order_id_clean") == "")
        | F.col("customer_id_clean").isNull()
        | (F.col("customer_id_clean") == "")
        | F.col("product_id_clean").isNull()
        | (F.col("product_id_clean") == "")
        | F.col("store_id_clean").isNull()
        | (F.col("store_id_clean") == "")
        | F.col("order_ts").isNull()
        | F.col("quantity").isNull()
        | F.col("unit_price").isNull()
        | F.col("gross_amount").isNull()
    )
)
print("Orders quarantine records:", order_quarantine.count())

print("\nQuarantine by reason:")

display(
    order_quarantine.select(
        F.when(
            F.col("product_id_clean").isNull() |
            (F.col("product_id_clean") == ""),
            "Missing product_id"
        ).when(
            F.col("customer_id_clean").isNull() |
            (F.col("customer_id_clean") == ""),
            "Missing customer_id"
        ).when(
            F.col("store_id_clean").isNull() |
            (F.col("store_id_clean") == ""),
            "Missing store_id"
        ).when(
            F.col("order_ts").isNull(),
            "Missing order_ts"
        ).when(
            F.col("quantity").isNull(),
            "Missing quantity"
        ).when(
            F.col("unit_price").isNull(),
            "Invalid unit_price"
        ).when(
            F.col("gross_amount").isNull(),
            "Invalid gross_amount"
        ).otherwise("Other")
        .alias("quarantine_reason")
    )
    .groupBy("quarantine_reason")
    .count()
    .orderBy(F.desc("count"))
)

Orders quarantine records: 284

Quarantine by reason:


quarantine_reason,count
Invalid unit_price,153
Invalid gross_amount,71
Missing product_id,60


In [0]:

#  WRITE ORDERS QUARANTINE + SILVER
(
    order_quarantine
    .write
    .format("delta")
    .mode("overwrite")
    .option("overwriteSchema", "true")
    .saveAsTable(
        f"{CATALOG}.{SILVER_SCHEMA}.quarantine_orders"
    )
)
silver_orders_final = (
    silver_orders
    .filter(
        F.col("order_id").isNotNull()
        & (F.trim(F.col("order_id")) != "")
        & F.col("customer_id").isNotNull()
        & (F.trim(F.col("customer_id")) != "")
        & F.col("product_id").isNotNull()
        & (F.trim(F.col("product_id")) != "")
        & F.col("store_id").isNotNull()
        & (F.trim(F.col("store_id")) != "")
        & F.col("order_ts").isNotNull()
        & F.col("quantity").isNotNull()
        & F.col("unit_price").isNotNull()
        & F.col("gross_amount").isNotNull()
    )
)

print("Trusted Orders rows:", silver_orders_final.count())

(
    silver_orders_final
    .write
    .format("delta")
    .mode("overwrite")
    .option("overwriteSchema", "true")
    .saveAsTable(
        f"{CATALOG}.{SILVER_SCHEMA}.silver1_orders_clean"
    )
)


display(
    spark.sql(f"""
        SELECT COUNT(*) AS row_count
        FROM {CATALOG}.{SILVER_SCHEMA}.silver1_orders_clean
    """)
)

Trusted Orders rows: 5716


row_count
5716


In [0]:
# LOAD & INSPECT STORES

bronze_stores = spark.table(
    f"{CATALOG}.{RAW_SCHEMA}.stores_bronze"
)
print("Total rows:", bronze_stores.count())
print(
    "Distinct store IDs:",
    bronze_stores.select("store_id").distinct().count()
)
print("\nSchema:")
bronze_stores.printSchema()
print("\nSample:")
display(bronze_stores.limit(20))

Total rows: 80
Distinct store IDs: 75

Schema:
root
 |-- store_id: string (nullable = true)
 |-- store_name: string (nullable = true)
 |-- city: string (nullable = true)
 |-- region: string (nullable = true)
 |-- status: string (nullable = true)
 |-- _ingested_at: timestamp (nullable = true)
 |-- _source_file: string (nullable = true)
 |-- _source_type: string (nullable = true)


Sample:


store_id,store_name,city,region,status,_ingested_at,_source_file,_source_type
S001,Store_1,Jaipur,Online,active,2026-08-12T17:22:10.106Z,stores_batch.csv,batch
S002,Store_2,Pune,West,active,2026-08-12T17:22:10.106Z,stores_batch.csv,batch
S003,Store_3,Ahmedabad,South,closed,2026-08-12T17:22:10.106Z,stores_batch.csv,batch
S004,Store_4,Ahmedabad,North,active,2026-08-12T17:22:10.106Z,stores_batch.csv,batch
S005,Store_5,Chennai,East,active,2026-08-12T17:22:10.106Z,stores_batch.csv,batch
S006,Store_6,Kolkata,East,closed,2026-08-12T17:22:10.106Z,stores_batch.csv,batch
S007,Store_7,Hyderabad,Online,active,2026-08-12T17:22:10.106Z,stores_batch.csv,batch
S008,Store_8,Chennai,North,closed,2026-08-12T17:22:10.106Z,stores_batch.csv,batch
S009,Store_9,Bengaluru,West,active,2026-08-12T17:22:10.106Z,stores_batch.csv,batch
S010,Store_10,Gurugram,East,active,2026-08-12T17:22:10.106Z,stores_batch.csv,batch


In [0]:
#  CLEAN & DEDUPLICATE STORES

silver_stores = (
    bronze_stores
    .select(
        "store_id",
        "store_name",
        "city",
        "region",
        "status",
        "_ingested_at",
        "_source_file",
        "_source_type"
    )
    # Clean identifiers / strings
    .withColumn("store_id", F.trim(F.col("store_id")))
    .withColumn("store_name", F.trim(F.col("store_name")))
    .withColumn("city", F.trim(F.col("city")))
    .withColumn("region", F.trim(F.col("region")))
    .withColumn("status", F.trim(F.col("status")))

    # Handle missing categorical values
    .withColumn(
        "region",
        F.when(
            F.col("region").isNull() |
            (F.trim(F.col("region")) == ""),
            F.lit("Unknown")
        ).otherwise(F.col("region"))
    )
    .withColumn(
        "status",
        F.when(
            F.col("status").isNull() |
            (F.trim(F.col("status")) == ""),
            F.lit("Unknown")
        ).otherwise(F.col("status"))
    )
    .withColumn(
        "rn",
        F.row_number().over(
            Window
            .partitionBy("store_id")
            .orderBy(F.col("_ingested_at").desc())
        )
    )
    .filter(F.col("rn") == 1)
    .drop("rn")
)

print("Stores Silver cleaning completed.")
print("Rows after deduplication:", silver_stores.count())

silver_stores.printSchema()

Stores Silver cleaning completed.
Rows after deduplication: 75
root
 |-- store_id: string (nullable = true)
 |-- store_name: string (nullable = true)
 |-- city: string (nullable = true)
 |-- region: string (nullable = true)
 |-- status: string (nullable = true)
 |-- _ingested_at: timestamp (nullable = true)
 |-- _source_file: string (nullable = true)
 |-- _source_type: string (nullable = true)



In [0]:
#  WRITE STORES SILVER
(
    silver_stores
    .write
    .format("delta")
    .mode("overwrite")
    .option("overwriteSchema", "true")
    .saveAsTable(
        f"{CATALOG}.{SILVER_SCHEMA}.silver1_stores_clean"
    )
)
print("Stores Silver table written successfully.")
display(
    spark.sql(f"""
        SELECT COUNT(*) AS row_count
        FROM {CATALOG}.{SILVER_SCHEMA}.silver1_stores_clean
    """)
)

Stores Silver table written successfully.


row_count
75


In [0]:
# FINAL SILVER STAGE 1 VALIDATION


silver_tables = [
    "silver1_customers_clean",
    "silver1_products_clean",
    "silver1_orders_clean",
    "silver1_stores_clean",
    "quarantine_orders"
]
for table_name in silver_tables:
    full_name = f"{CATALOG}.{SILVER_SCHEMA}.{table_name}"
    try:
        count = spark.table(full_name).count()
        print(f"✓ {full_name:<60} {count:,} rows")
    except Exception:
        print(f"✗ {full_name:<60} NOT FOUND")

✓ retail_demo.silver.silver1_customers_clean                   577 rows
✓ retail_demo.silver.silver1_products_clean                    166 rows
✓ retail_demo.silver.silver1_orders_clean                      5,716 rows
✓ retail_demo.silver.silver1_stores_clean                      75 rows
✓ retail_demo.silver.quarantine_orders                         284 rows
